# FHIR Data Platform — Gold Layer Notebook
## Star Schema Build using PySpark
| | |
|---|---|
| **Source** | `fhir_data.prd_silver` (patient, encounter, observation, condition) |
| **Target** | `fhir_data.prd_gold` |
| **Pattern** | Star Schema — 5 Dims · 3 Facts · 5 Aggregations |
| **Date** | 2026-04-22 |

### Cell 1 — Imports & SparkSession

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, DateType, BooleanType
)
from datetime import datetime
RUN_DATE = datetime.now().strftime("%Y-%m-%d")
print("Run date:", RUN_DATE)

### Cell 2 — Load Silver Tables

In [0]:
patient_df     = spark.table("fhir_data.prd_silver.patient")
encounter_df   = spark.table("fhir_data.prd_silver.encounter")
observation_df = spark.table("fhir_data.prd_silver.observation")
condition_df   = spark.table("fhir_data.prd_silver.condition")

for name, df in [("patient", patient_df), ("encounter", encounter_df),
                 ("observation", observation_df), ("condition", condition_df)]:
    print(f"  {name:<12}: {df.count():,} rows")

### Cell 3 — `dim_patient` (SCD-1 Snapshot)
One row per patient. `Resource_id` kept as STRING — silver data contains non-numeric IDs (e.g. `lilou2`).

In [0]:
dim_patient = (
    patient_df
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("Resource_id").orderBy(F.col("Last_updated").desc())
    ))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("Resource_id").cast(StringType())       .alias("patient_key"),
        F.col("Resource_id").cast(StringType())       .alias("patient_id"),
        F.col("Gender")                               .alias("gender"),
        F.to_date("Birth_Date", "yyyy-MM-dd")         .alias("birth_date"),
        F.col("Age").cast(IntegerType())              .alias("age"),
        F.col("Age_Group")                            .alias("age_group"),
        F.coalesce("First_name", F.lit("Unknown"))    .alias("first_name"),
        F.coalesce("Last_name",  F.lit("Unknown"))    .alias("last_name"),
        F.col("Phone_Number")                         .alias("phone_number"),
        F.col("Email")                                .alias("email"),
        F.col("Active").cast(BooleanType())           .alias("is_active"),
        F.col("Source")                               .alias("source_system"),
        F.to_timestamp("Last_updated")                .alias("last_updated_ts"),
        F.lit(RUN_DATE).cast(DateType())              .alias("dw_load_date"),
    )
    .drop("_rn")
)
print("dim_patient:", dim_patient.count(), "rows")

### Cell 4 — `dim_date` (Date Spine 2015–2030)

In [0]:
dim_date = (
    spark.range(0, (datetime(2030,12,31) - datetime(2015,1,1)).days + 1)
    .select(F.expr("date_add(to_date('2015-01-01'), cast(id as int))").alias("full_date"))
    .select(
        F.date_format("full_date","yyyyMMdd").cast(IntegerType()) .alias("date_key"),
        F.col("full_date"),
        F.year("full_date")                                       .alias("year"),
        F.quarter("full_date")                                    .alias("quarter"),
        F.month("full_date")                                      .alias("month"),
        F.date_format("full_date","MMMM")                         .alias("month_name"),
        F.dayofmonth("full_date")                                 .alias("day_of_month"),
        F.dayofweek("full_date")                                  .alias("day_of_week"),
        F.date_format("full_date","EEEE")                         .alias("day_name"),
        F.weekofyear("full_date")                                 .alias("week_of_year"),
        F.concat(F.lit("Q"), F.quarter("full_date"),
                 F.lit("-"), F.year("full_date"))                 .alias("quarter_label"),
        F.dayofweek("full_date").isin([1,7]).cast(BooleanType())  .alias("is_weekend"),
    )
)
print("dim_date:", dim_date.count(), "rows")

### Cell 5 — `dim_encounter`
Latest record per encounter deduped via row_number.

In [0]:
dim_encounter = (
    encounter_df
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("encounter_id").orderBy(F.col("last_updated_ts").desc())
    ))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("encounter_id")                              .alias("encounter_key"),
        F.col("encounter_id"),
        F.col("patient_id"),
        F.col("status")                                    .alias("encounter_status"),
        F.col("resource_type"),
        F.col("identifier_system"),
        F.col("identifier_type"),
        F.col("identifier_numeric"),
        F.col("organization_id"),
        F.col("source"),
        F.to_timestamp("last_updated_ts")                  .alias("last_updated_ts"),
        F.to_date("ingestion_date","yyyy-MM-dd")           .alias("ingestion_date"),
        F.lit(RUN_DATE).cast(DateType())                   .alias("dw_load_date"),
    )
    .drop("_rn")
)
print("dim_encounter:", dim_encounter.count(), "rows")

### Cell 6 — `dim_observation_type` (LOINC Code Lookup)

In [0]:
dim_observation_type = (
    observation_df
    .select(
        F.col("code")              .alias("observation_code"),
        F.col("code_text_clean")   .alias("observation_name"),
        F.col("observation_type"),
        F.col("data_source_type"),
        F.col("unit_clean")        .alias("default_unit"),
        F.col("unit_system_clean") .alias("unit_system"),
    )
    .dropDuplicates(["observation_code"])
    .withColumn("observation_type_key",
        F.dense_rank().over(Window.orderBy("observation_code")))
)
print("dim_observation_type:", dim_observation_type.count(), "rows")

### Cell 7 — `dim_condition_type` (ICD-10 Code Lookup)

In [0]:
dim_condition_type = (
    condition_df
    .select(
        F.col("condition_code"),
        F.col("condition_display"),
        F.col("coding_system"),
        F.col("resource_type"),
    )
    .dropDuplicates(["condition_code"])
    .withColumn("condition_type_key",
        F.dense_rank().over(Window.orderBy("condition_code")))
)
print("dim_condition_type:", dim_condition_type.count(), "rows")

### Cell 8 — `fact_patient_encounter`
Grain: one row per patient × encounter.

In [0]:
fact_patient_encounter = (
    encounter_df
    .join(
        patient_df.select(
            F.col("Resource_id").cast(StringType()).alias("patient_id"),
            "Age", "Age_Group", "Gender",
        ),
        on="patient_id", how="left",
    )
    .select(
        F.col("encounter_id")                                     .alias("encounter_key"),
        F.col("patient_id"),
        F.date_format(
            F.to_date("ingestion_date","yyyy-MM-dd"),"yyyyMMdd"
        ).cast(IntegerType())                                     .alias("date_key"),
        F.col("status")                                           .alias("encounter_status"),
        F.col("Age").cast(IntegerType())                          .alias("patient_age_at_encounter"),
        F.col("Age_Group")                                        .alias("patient_age_group"),
        F.col("Gender")                                           .alias("patient_gender"),
        F.col("source")                                           .alias("source_system"),
        F.col("version_id"),
        (F.col("status") == "finished").cast(IntegerType())       .alias("is_finished"),
        F.lit(1).cast(IntegerType())                              .alias("encounter_count"),
        F.to_date("ingestion_date","yyyy-MM-dd")                  .alias("ingestion_date"),
        F.lit(RUN_DATE).cast(DateType())                          .alias("dw_load_date"),
    )
)
print("fact_patient_encounter:", fact_patient_encounter.count(), "rows")

### Cell 9 — `fact_observation`
Grain: one row per clinical observation measurement. `try_cast` used for IDs that may contain non-numeric values.

In [0]:
fact_observation = (
    observation_df
    .join(
        dim_observation_type.select("observation_code","observation_type_key"),
        observation_df.code == dim_observation_type.observation_code,
        how="left",
    )
    .select(
        F.expr("try_cast(observation_id as BIGINT)")       .alias("observation_key"),
        F.col("patient_id").cast(StringType()),
        F.coalesce(F.col("encounter_id").cast(StringType()),
                   F.lit("NO_ENCOUNTER"))                  .alias("encounter_key"),
        F.col("observation_type_key"),
        F.col("code")                                      .alias("observation_code"),
        F.col("code_text_clean")                           .alias("observation_name"),
        F.col("observation_type"),
        F.col("data_source_type"),
        F.col("observation_value").cast(FloatType()),
        F.col("unit_clean")                                .alias("unit"),
        F.col("status"),
        F.date_format(
            F.to_date(F.substring("effective_ts",1,10),"yyyy-MM-dd"),"yyyyMMdd"
        ).cast(IntegerType())                              .alias("effective_date_key"),
        F.date_format(
            F.to_date("ingestion_date","yyyy-MM-dd"),"yyyyMMdd"
        ).cast(IntegerType())                              .alias("ingestion_date_key"),
        F.to_timestamp("effective_ts")                     .alias("effective_ts"),
        F.lit(1).cast(IntegerType())                       .alias("observation_count"),
        F.lit(RUN_DATE).cast(DateType())                   .alias("dw_load_date"),
    )
)
print("fact_observation:", fact_observation.count(), "rows")

### Cell 10 — `fact_condition`
Grain: one row per patient condition diagnosis. Join pulls only `condition_type_key` from dim to avoid ambiguous `condition_display` / `coding_system` columns.

In [0]:
_dim_cond_key = dim_condition_type.select(
    F.col("condition_code").alias("_dim_condition_code"),
    F.col("condition_type_key"),
)

fact_condition = (
    condition_df
    .join(_dim_cond_key,
          condition_df.condition_code == _dim_cond_key._dim_condition_code,
          how="left")
    .select(
        F.expr("try_cast(condition_id as BIGINT)")         .alias("condition_key"),
        condition_df["patient_id"].cast(StringType())      .alias("patient_id"),
        F.coalesce(condition_df["encounter_id"].cast(StringType()),
                   F.lit("NO_ENCOUNTER"))                  .alias("encounter_key"),
        F.col("condition_type_key"),
        condition_df["condition_code"],
        condition_df["condition_display"],
        condition_df["coding_system"],
        condition_df["clinical_status"],
        (condition_df["clinical_status"] == "active")
            .cast(IntegerType())                           .alias("is_active_condition"),
        F.col("dq_flags"),
        F.date_format(
            F.to_date(condition_df["ingestion_date"],"yyyy-MM-dd"),"yyyyMMdd"
        ).cast(IntegerType())                              .alias("ingestion_date_key"),
        F.lit(1).cast(IntegerType())                       .alias("condition_count"),
        F.lit(RUN_DATE).cast(DateType())                   .alias("dw_load_date"),
    )
)
print("fact_condition:", fact_condition.count(), "rows")

### Cell 11 — `agg_patient_summary`
One row per patient — encounters, conditions, and average vitals rolled up from all domains.

In [0]:
enc_agg = (
    fact_patient_encounter.groupBy("patient_id")
    .agg(
        F.count("encounter_key")          .alias("total_encounters"),
        F.sum("is_finished")              .alias("finished_encounters"),
        F.max("ingestion_date")           .alias("last_encounter_date"),
    )
)

cond_agg = (
    fact_condition.groupBy("patient_id")
    .agg(
        F.count("condition_key")          .alias("total_conditions"),
        F.sum("is_active_condition")      .alias("active_conditions"),
        F.collect_set("condition_display").alias("condition_list"),
    )
)

obs_agg = (
    observation_df.groupBy("patient_id")
    .agg(
        F.count("*")                                                                    .alias("total_observations"),
        F.round(F.avg(F.when(F.col("code")=="8867-4",  F.col("observation_value"))),2) .alias("avg_heart_rate_bpm"),
        F.round(F.avg(F.when(F.col("code")=="8462-4",  F.col("observation_value"))),2) .alias("avg_diastolic_bp_mmhg"),
        F.round(F.avg(F.when(F.col("code")=="8480-6",  F.col("observation_value"))),2) .alias("avg_systolic_bp_mmhg"),
        F.round(F.avg(F.when(F.col("code")=="29463-7", F.col("observation_value"))),2) .alias("avg_body_weight_kg"),
        F.round(F.avg(F.when(F.col("code")=="8302-2",  F.col("observation_value"))),2) .alias("avg_body_height_cm"),
    )
)

agg_patient_summary = (
    dim_patient
    .join(enc_agg,  on="patient_id", how="left")
    .join(cond_agg, on="patient_id", how="left")
    .join(obs_agg,  on="patient_id", how="left")
    .select(
        "patient_key","patient_id","gender","birth_date","age","age_group",
        "first_name","last_name","is_active","source_system",
        F.coalesce("total_encounters",    F.lit(0)).alias("total_encounters"),
        F.coalesce("finished_encounters", F.lit(0)).alias("finished_encounters"),
        "last_encounter_date",
        F.coalesce("total_conditions",    F.lit(0)).alias("total_conditions"),
        F.coalesce("active_conditions",   F.lit(0)).alias("active_conditions"),
        "condition_list",
        F.coalesce("total_observations",  F.lit(0)).alias("total_observations"),
        "avg_heart_rate_bpm","avg_diastolic_bp_mmhg","avg_systolic_bp_mmhg",
        "avg_body_weight_kg","avg_body_height_cm",
        F.lit(RUN_DATE).cast(DateType()).alias("dw_load_date"),
    )
)
print("agg_patient_summary:", agg_patient_summary.count(), "rows")

### Cell 12 — `agg_condition_prevalence`
ICD-10 condition prevalence % ranked across the patient population.

In [0]:
total_patients = dim_patient.count()

agg_condition_prevalence = (
    fact_condition
    .groupBy("condition_code","condition_display","coding_system","clinical_status")
    .agg(
        F.countDistinct("patient_id").alias("patient_count"),
        F.count("condition_key")     .alias("total_diagnoses"),
    )
    .withColumn("prevalence_pct",
        F.round(F.col("patient_count") / F.lit(total_patients) * 100, 2))
    .orderBy(F.col("patient_count").desc())
    .withColumn("dw_load_date", F.lit(RUN_DATE).cast(DateType()))
)
print("agg_condition_prevalence:", agg_condition_prevalence.count(), "rows")

### Cell 13 — `agg_observation_stats`
Statistical profile (avg, min, max, stddev, P25/P50/P75) per LOINC observation type.

In [0]:
agg_observation_stats = (
    observation_df
    .filter(F.col("observation_value").isNotNull())
    .groupBy("code","code_text_clean","observation_type","unit_clean")
    .agg(
        F.count("*")                                               .alias("reading_count"),
        F.countDistinct("patient_id")                             .alias("patient_count"),
        F.round(F.avg("observation_value"),    2)                 .alias("avg_value"),
        F.round(F.min("observation_value"),    2)                 .alias("min_value"),
        F.round(F.max("observation_value"),    2)                 .alias("max_value"),
        F.round(F.stddev("observation_value"), 2)                 .alias("stddev_value"),
        F.round(F.expr("percentile(observation_value,0.25)"), 2)  .alias("p25_value"),
        F.round(F.expr("percentile(observation_value,0.50)"), 2)  .alias("median_value"),
        F.round(F.expr("percentile(observation_value,0.75)"), 2)  .alias("p75_value"),
    )
    .withColumnRenamed("code",            "observation_code")
    .withColumnRenamed("code_text_clean", "observation_name")
    .withColumnRenamed("unit_clean",      "unit")
    .withColumn("dw_load_date", F.lit(RUN_DATE).cast(DateType()))
)
print("agg_observation_stats:", agg_observation_stats.count(), "rows")

### Cell 14 — `agg_encounter_by_age_gender`
Encounter volume sliced by patient age group and gender.

In [0]:
agg_encounter_by_age_gender = (
    fact_patient_encounter
    .groupBy("patient_age_group","patient_gender","encounter_status")
    .agg(
        F.count("encounter_key")       .alias("encounter_count"),
        F.countDistinct("patient_id")  .alias("unique_patients"),
    )
    .orderBy("patient_age_group","patient_gender")
    .withColumn("dw_load_date", F.lit(RUN_DATE).cast(DateType()))
)
print("agg_encounter_by_age_gender:", agg_encounter_by_age_gender.count(), "rows")

### Cell 15 — `agg_monthly_activity`
Monthly ingestion trend of encounters and observations.

In [0]:
monthly_enc = (
    fact_patient_encounter
    .withColumn("year_month", F.date_format("ingestion_date","yyyy-MM"))
    .groupBy("year_month")
    .agg(
        F.count("encounter_key")       .alias("encounter_count"),
        F.countDistinct("patient_id")  .alias("unique_patients"),
    )
)

monthly_obs = (
    observation_df
    .withColumn("year_month",
        F.date_format(F.to_date("ingestion_date","yyyy-MM-dd"),"yyyy-MM"))
    .groupBy("year_month")
    .agg(F.count("*").alias("observation_count"))
)

agg_monthly_activity = (
    monthly_enc.join(monthly_obs, on="year_month", how="outer")
    .orderBy("year_month")
    .withColumn("dw_load_date", F.lit(RUN_DATE).cast(DateType()))
)
print("agg_monthly_activity:", agg_monthly_activity.count(), "rows")

### Cell 16 — Write All Gold Tables
Writes all 13 tables to `fhir_data.prd_gold` in Delta format with `overwriteSchema=true`.

In [0]:
GOLD_DB = "fhir_data.prd_gold"

def write_gold(df, table_name, partition_col=None):
    full_name = f"{GOLD_DB}.{table_name}"
    writer = (
        df.write.format("delta")
          .mode("append")
          .option("MergeSchema","true")
    )
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(full_name)
    cnt = spark.table(full_name).count()
    print(f"  OK  {full_name:<55}  {cnt:>6,} rows")

print("--- Dimension Tables ---")
write_gold(dim_patient,            "dim_patient")
write_gold(dim_date,               "dim_date")
write_gold(dim_encounter,          "dim_encounter")
write_gold(dim_observation_type,   "dim_observation_type")
write_gold(dim_condition_type,     "dim_condition_type")

print("\n--- Fact Tables ---")
write_gold(fact_patient_encounter, "fact_patient_encounter", "ingestion_date")
write_gold(fact_observation,       "fact_observation",       "dw_load_date")
write_gold(fact_condition,         "fact_condition",         "dw_load_date")

print("\n--- Aggregation Tables ---")
write_gold(agg_patient_summary,          "agg_patient_summary")
write_gold(agg_condition_prevalence,     "agg_condition_prevalence")
write_gold(agg_observation_stats,        "agg_observation_stats")
write_gold(agg_encounter_by_age_gender,  "agg_encounter_by_age_gender")
write_gold(agg_monthly_activity,         "agg_monthly_activity")

print("\nGold layer load complete!")

### Cell 17 — Validation
Row counts and NULL checks on primary keys for all 13 gold tables.

In [0]:
print("GOLD LAYER VALIDATION")
print("-" * 65)

gold_tables = [
    ("dim_patient",                "patient_key"),
    ("dim_date",                   "date_key"),
    ("dim_encounter",              "encounter_key"),
    ("dim_observation_type",       "observation_type_key"),
    ("dim_condition_type",         "condition_type_key"),
    ("fact_patient_encounter",     "encounter_key"),
    ("fact_observation",           "observation_key"),
    ("fact_condition",             "condition_key"),
    ("agg_patient_summary",        "patient_key"),
    ("agg_condition_prevalence",   "condition_code"),
    ("agg_observation_stats",      "observation_code"),
    ("agg_encounter_by_age_gender","encounter_count"),
    ("agg_monthly_activity",       "year_month"),
]

for tbl, pk in gold_tables:
    df_v  = spark.table(f"{GOLD_DB}.{tbl}")
    total = df_v.count()
    nulls = df_v.filter(F.col(pk).isNull()).count()
    print(f"  {tbl:<44}  rows={total:>6,}  null_{pk}={nulls}")

### Cell 18 — Sample: Top 10 Most Prevalent Conditions

In [0]:
spark.table(f"{GOLD_DB}.agg_condition_prevalence") \
    .select("condition_code","condition_display","patient_count","prevalence_pct") \
    .orderBy(F.col("patient_count").desc()) \
    .limit(10) \
    .show(truncate=False)

### Cell 19 — Sample: Patient Summary (first 5 rows)

In [0]:
spark.table(f"{GOLD_DB}.agg_patient_summary") \
    .select("patient_id","gender","age","age_group",
            "total_encounters","active_conditions",
            "avg_heart_rate_bpm","avg_diastolic_bp_mmhg") \
    .limit(5) \
    .show(truncate=False)

### Cell 20 — Sample: Observation Statistics

In [0]:
spark.table(f"{GOLD_DB}.agg_observation_stats") \
    .select("observation_name","unit","patient_count","reading_count",
            "avg_value","min_value","max_value","median_value") \
    .show(truncate=False)